# ASCII Maze RL — SFT Training

This notebook trains a LoRA adapter on solved maze examples using SFT
(Supervised Fine-Tuning). The resulting adapter is pushed to HuggingFace Hub
and used as the starting point for the GRPO workshop notebook.

Run on Google Colab with a **T4 GPU** runtime.

## What this produces
- A LoRA adapter that teaches the model to read maze grids and output solutions
- The model learns the format (space-separated u/d/l/r moves) and basic maze-solving
- GRPO then refines accuracy on top of this foundation

In [ ]:
# Install deps — avoid overwriting Colab's CUDA PyTorch
!pip install -q trl peft datasets accelerate bitsandbytes 2>&1 | tail -1

# Verify CUDA torch is still intact
import torch
if not torch.cuda.is_available():
    print("⚠️  CUDA not available! Restart runtime: Runtime → Restart session")
    print("   Then re-run all cells from the top.")
else:
    print(f"✓ CUDA OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the maze infrastructure
!git clone -q -b claude-trl https://github.com/StephenJHardy/ascii-maze-rl.git
import sys
sys.path.insert(0, 'ascii-maze-rl')

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")

## 1. Generate Training Data

In [ ]:
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig
from src.maze_repr import SYSTEM_PROMPT

# Training data: solved maze examples across sizes
# More data = better, especially for larger mazes
train_config = DatasetConfig(
    name="sft_train",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=3, height=3, count=5000, start_seed=700_000),
        SizeConfig(width=4, height=4, count=10000, start_seed=710_000),
        SizeConfig(width=5, height=5, count=15000, start_seed=720_000),
        SizeConfig(width=6, height=6, count=12000, start_seed=730_000),
        SizeConfig(width=7, height=7, count=8000, start_seed=740_000),
    ],
)
train_ds = MazeDataset.generate(train_config, progress=True)
print(train_ds.summary())

# Eval data (disjoint seeds)
eval_config = DatasetConfig(
    name="sft_eval",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=3, height=3, count=50, start_seed=900_000),
        SizeConfig(width=4, height=4, count=50, start_seed=910_000),
        SizeConfig(width=5, height=5, count=50, start_seed=920_000),
        SizeConfig(width=6, height=6, count=30, start_seed=930_000),
        SizeConfig(width=7, height=7, count=20, start_seed=940_000),
    ],
)
eval_ds = MazeDataset.generate(eval_config, progress=False)
print(eval_ds.summary())

In [ ]:
from datasets import Dataset

def records_to_chat_dataset(records):
    """Convert maze records to chat format for SFT."""
    rows = []
    for r in records:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": r.maze_str},
            {"role": "assistant", "content": r.solution_moves},
        ]
        rows.append({"messages": messages})
    return Dataset.from_list(rows)

train_hf = records_to_chat_dataset(train_ds.records)
eval_hf = records_to_chat_dataset(eval_ds.records)
print(f"Training examples: {len(train_hf)}")
print(f"Eval examples: {len(eval_hf)}")
print(f"\nSample:\n{train_hf[0]['messages']}")

## 2. Load Model and Configure LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

print(f"Model: {model_id}")
print(f"LoRA rank: {peft_config.r}")
print(f"Target modules: {peft_config.target_modules}")

## 2b. Baseline Evaluation (before SFT)

Let's see how the base model performs on mazes before any fine-tuning.
This gives us the starting point to measure SFT's impact.

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str
from src.maze_verify import extract_moves, simulate, reached_exit

def generate_solution(mdl, tok, maze, max_new_tokens=40):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": to_str(maze)},
    ]
    text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True,
    )

# Evaluate base model
print("Evaluating base model (before SFT)...")
base_results = {}
total = len(eval_ds.records)
solved_total = 0

for idx, record in enumerate(eval_ds):
    maze = record.to_maze()
    response = generate_solution(model, tokenizer, maze)
    moves = extract_moves(response)
    path = simulate(moves or [], maze)
    solved = reached_exit(path, maze)
    size = f"{record.width}x{record.height}"
    if size not in base_results:
        base_results[size] = {"total": 0, "solved": 0}
    base_results[size]["total"] += 1
    base_results[size]["solved"] += int(solved)
    solved_total += int(solved)

    if (idx + 1) % 20 == 0 or idx == total - 1:
        pct = solved_total / (idx + 1) * 100
        print(f"  [{idx+1}/{total}] solved={solved_total}/{idx+1} ({pct:.0f}%)")

print("\nBase Model Results (before SFT):")
for size, stats in sorted(base_results.items()):
    rate = stats['solved'] / stats['total'] * 100
    print(f"  {size}: {stats['solved']}/{stats['total']} ({rate:.0f}%)")

# Show some sample outputs
print("\nSample base model outputs:")
for seed in [42, 99]:
    maze = generate(4, 4, seed=seed)
    response = generate_solution(model, tokenizer, maze)
    print(f"  4x4_{seed}: {response!r:.60s}")

## 3. Train with SFT

In [ ]:
from trl import SFTConfig, SFTTrainer

# Training config — adjust max_steps for longer/shorter training
# 2000 steps ≈ 20-30 min on T4, gives good results
# 5000 steps ≈ 60-90 min on T4, gives best results
sft_config = SFTConfig(
    output_dir="./sft-output",
    max_steps=2000,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    eval_strategy="steps",
    bf16=True,
    report_to="none",
    gradient_accumulation_steps=1,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_hf,
    eval_dataset=eval_hf,
    peft_config=peft_config,
)

print(f"Training for {sft_config.max_steps} steps...")

In [ ]:
trainer.train()

## 4. Evaluate

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str
from src.maze_verify import extract_moves, simulate, reached_exit

def generate_solution(model, tokenizer, maze, max_new_tokens=40):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": to_str(maze)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True,
    )

# Evaluate
trained_model = trainer.model
results = {}
total = len(eval_ds.records)
solved_total = 0

for idx, record in enumerate(eval_ds):
    maze = record.to_maze()
    response = generate_solution(trained_model, tokenizer, maze)
    moves = extract_moves(response)
    path = simulate(moves or [], maze)
    solved = reached_exit(path, maze)
    size = f"{record.width}x{record.height}"
    if size not in results:
        results[size] = {"total": 0, "solved": 0}
    results[size]["total"] += 1
    results[size]["solved"] += int(solved)
    solved_total += int(solved)

    if (idx + 1) % 20 == 0 or idx == total - 1:
        pct = solved_total / (idx + 1) * 100
        print(f"  [{idx+1}/{total}] solved={solved_total}/{idx+1} ({pct:.0f}%)")

print("\nSFT Results:")
for size, stats in sorted(results.items()):
    rate = stats['solved'] / stats['total'] * 100
    print(f"  {size}: {stats['solved']}/{stats['total']} ({rate:.0f}%)")

# Compare before/after
print("\nBefore vs After SFT:")
print(f"{'Size':>5s}  {'Before':>8s}  {'After':>8s}  {'Change':>8s}")
print("-" * 35)
for size in sorted(set(list(base_results.keys()) + list(results.keys()))):
    pre = base_results.get(size, {'solved': 0, 'total': 1})
    post = results.get(size, {'solved': 0, 'total': 1})
    pre_rate = pre['solved'] / pre['total'] * 100
    post_rate = post['solved'] / post['total'] * 100
    diff = post_rate - pre_rate
    print(f"{size:>5s}  {pre_rate:7.1f}%  {post_rate:7.1f}%  {diff:+7.1f}pp")

In [ ]:
# Look at some examples
print("Sample outputs:\n")
for seed in [42, 99, 7]:
    for size in [3, 4, 5]:
        maze = generate(size, size, seed=seed)
        response = generate_solution(trained_model, tokenizer, maze)
        from src.maze_repr import solution_to_str
        correct = solution_to_str(maze.solution_moves)
        moves = extract_moves(response)
        path = simulate(moves or [], maze)
        solved = reached_exit(path, maze)
        status = "OK" if solved else "XX"
        print(f"  {status} {size}x{size}_{seed}: model={response!r:30s} correct={correct}")

## 5. Push to HuggingFace Hub

Save the adapter so the GRPO workshop notebook can load it.

In [ ]:
!pip install -q ipywidgets
from huggingface_hub import login
login()  # Enter your HF token with write access

In [ ]:
# Change this to your HuggingFace username
hub_repo = "StephenJHardy/maze-cuda-sft-qwen2.5-0.5b"

# Merge LoRA into base model for clean loading
merged_model = trainer.model.merge_and_unload()

merged_model.push_to_hub(hub_repo)
tokenizer.push_to_hub(hub_repo)
print(f"\nPushed merged model to https://huggingface.co/{hub_repo}")
print("The GRPO notebook can load this directly — no adapter needed.")